# Обучение модели классификации IT-ролей

Ноутбук посвящен обучению модели, которая определяет конкретную IT-роль кандидата по тексту резюме.


## Импорты и настройка окружения


In [3]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC


warnings.filterwarnings("ignore")


if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

Device: cuda
CUDA available: True


In [18]:
BASE_DIR = Path(".").resolve()

DATA_DIR = BASE_DIR / "data" / "processed"
DATA_PATH = BASE_DIR / "data" / "processed" / "resume_dataset_it_roles.csv"
MODEL_DIR = BASE_DIR / "app" / "models" / "it_role_model"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

TEXT_COLUMN = "resume_text"
TARGET_COLUMN = "target_role"

print("Dataset path:", DATA_PATH)
print("Model directory:", MODEL_DIR)

Dataset path: /content/data/processed/resume_dataset_it_roles.csv
Model directory: /content/app/models/it_role_model


## Загрузка датасета


In [6]:
raw_df = pd.read_csv(DATA_PATH)

print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
print()

print("Target roles:")
print(raw_df[TARGET_COLUMN].value_counts())
print()

print("Sources:")
print(raw_df["source"].value_counts())

raw_df.head()

Shape: (8608, 4)
Columns: ['resume_text', 'target_role', 'source', 'original_category']

Target roles:
target_role
Python Developer          1284
Java Developer            1219
Web Developer              972
Database Administrator     931
Security Engineer          879
Systems Administrator      769
Project Manager IT         725
Frontend Developer         523
Network Administrator      469
Software Developer         437
SAP Developer               50
Blockchain Developer        50
QA Engineer                 50
DotNet Developer            50
Data Scientist              50
Business Analyst IT         50
DevOps Engineer             50
Data Engineer               50
Name: count, dtype: int64

Sources:
source
kaggle_avishekmajhi_resume_dataset    8166
synthetic_balancing_template           333
kaggle_updated_resume_dataset          109
Name: count, dtype: int64


,resume_text,target_role,source,original_category
0,Education Details\nMay 2013 Master Computer Ap...,SAP Developer,kaggle_updated_resume_dataset,SAP Developer
1,Java Developer Java Developer Java Developer -...,Java Developer,kaggle_avishekmajhi_resume_dataset,Java_Developer
2,Systems Administrator/ Network Administrator S...,Systems Administrator,kaggle_avishekmajhi_resume_dataset,Systems_Administrator
3,"Issues Management Analyst, GRC Issues Manageme...",Security Engineer,kaggle_avishekmajhi_resume_dataset,Security_Analyst
4,NETWORK ENGINEER/IT ADMINISTRATOR NETWORK ENGI...,Network Administrator,kaggle_avishekmajhi_resume_dataset,Network_Administrator


## Очистка данных

- проверяется наличие обязательных колонок;
- удаляются строки с пропущенными значениями;
- текстовые поля приводятся к строковому типу и очищаются от лишних пробелов;
- удаляются слишком короткие резюме;
- удаляются дубликаты по паре `resume_text` и `target_role`.


In [8]:
required_columns = {TEXT_COLUMN, TARGET_COLUMN, "source", "original_category"}
missing_columns = required_columns - set(raw_df.columns)

df = raw_df[[TEXT_COLUMN, TARGET_COLUMN, "source", "original_category"]].dropna()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()
df["source"] = df["source"].astype(str).str.strip()
df["original_category"] = df["original_category"].astype(str).str.strip()

df = df[df[TEXT_COLUMN].str.len() > 100]
df = df.drop_duplicates(subset=[TEXT_COLUMN, TARGET_COLUMN])

print("Prepared shape:", df.shape)
print()
print("Prepared class distribution:")
print(df[TARGET_COLUMN].value_counts())

df.head()

Prepared shape: (8608, 4)

Prepared class distribution:
target_role
Python Developer          1284
Java Developer            1219
Web Developer              972
Database Administrator     931
Security Engineer          879
Systems Administrator      769
Project Manager IT         725
Frontend Developer         523
Network Administrator      469
Software Developer         437
SAP Developer               50
Blockchain Developer        50
QA Engineer                 50
DotNet Developer            50
Data Scientist              50
Business Analyst IT         50
DevOps Engineer             50
Data Engineer               50
Name: count, dtype: int64


,resume_text,target_role,source,original_category
0,Education Details\nMay 2013 Master Computer Ap...,SAP Developer,kaggle_updated_resume_dataset,SAP Developer
1,Java Developer Java Developer Java Developer -...,Java Developer,kaggle_avishekmajhi_resume_dataset,Java_Developer
2,Systems Administrator/ Network Administrator S...,Systems Administrator,kaggle_avishekmajhi_resume_dataset,Systems_Administrator
3,"Issues Management Analyst, GRC Issues Manageme...",Security Engineer,kaggle_avishekmajhi_resume_dataset,Security_Analyst
4,NETWORK ENGINEER/IT ADMINISTRATOR NETWORK ENGI...,Network Administrator,kaggle_avishekmajhi_resume_dataset,Network_Administrator


## Подготовка train/test split


In [9]:
texts = df[TEXT_COLUMN].tolist()
labels = df[TARGET_COLUMN].tolist()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", len(X_train_texts))
print("Test:", len(X_test_texts))
print("Classes:", label_encoder.classes_.tolist())

Train: 6886
Test: 1722
Classes: ['Blockchain Developer', 'Business Analyst IT', 'Data Engineer', 'Data Scientist', 'Database Administrator', 'DevOps Engineer', 'DotNet Developer', 'Frontend Developer', 'Java Developer', 'Network Administrator', 'Project Manager IT', 'Python Developer', 'QA Engineer', 'SAP Developer', 'Security Engineer', 'Software Developer', 'Systems Administrator', 'Web Developer']


## Построение embeddings

Для преобразования текстов резюме в числовые признаки используется модель `BAAI/bge-small-en-v1.5`.


In [10]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device,
)


def encode_texts(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=64 if device == "cuda" else 32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=device,
    )


print("Encoding train texts")
X_train_emb = encode_texts(embedding_model, X_train_texts)

print("Encoding test texts")
X_test_emb = encode_texts(embedding_model, X_test_texts)

print("Train embeddings:", X_train_emb.shape)
print("Test embeddings:", X_test_emb.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

Encoding train texts...


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

Encoding test texts...


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Train embeddings: (6886, 384)
Test embeddings: (1722, 384)


## Функция оценки качества


In [11]:
def evaluate_predictions(
    model_name: str,
    y_true_encoded: np.ndarray,
    y_pred_encoded: np.ndarray,
    label_encoder: LabelEncoder,
) -> dict:
    y_true_labels = label_encoder.inverse_transform(y_true_encoded)
    y_pred_labels = label_encoder.inverse_transform(y_pred_encoded)

    metrics = {
        "model_name": model_name,
        "accuracy": accuracy_score(y_true_labels, y_pred_labels),
        "macro_f1": f1_score(y_true_labels, y_pred_labels, average="macro"),
        "weighted_f1": f1_score(y_true_labels, y_pred_labels, average="weighted"),
        "precision_macro": precision_score(
            y_true_labels,
            y_pred_labels,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_true_labels,
            y_pred_labels,
            average="macro",
            zero_division=0,
        ),
    }

    print(model_name)
    print(metrics)
    print('\n', classification_report(y_true_labels, y_pred_labels, zero_division=0))

    return metrics

## Сравнение классификаторов

После построения embeddings сравниваются два линейных классификатора:

- `LogisticRegression`
- `LinearSVC`


In [24]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.svm import LinearSVC

In [28]:
scoring = make_scorer(f1_score, average="macro")

candidate_models = {
    "LogisticRegression": {
        "model": LogisticRegression(
            solver="lbfgs",
            max_iter=5000,
            random_state=42,
            n_jobs=-1,
        ),
        "params": {
            "C": [0.3, 1, 3, 10, 30],
            "class_weight": [None, "balanced"],
        },
    },
    "LinearSVC": {
        "model": LinearSVC(
            random_state=42,
        ),
        "params": {
            "C": [0.3, 1, 3, 10],
            "class_weight": [None, "balanced"],
            "loss": ["squared_hinge"],
        },
    },
}

tuning_results = []
best_estimators = {}

for model_name, config in candidate_models.items():
    print(f"Tuning {model_name}")

    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        scoring=scoring,
        cv=5,
        n_jobs=-1,
        verbose=1,
    )

    grid.fit(X_train_emb, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_emb)

    metrics = evaluate_predictions(
        model_name=f"IT Role Classifier: BGE + tuned {model_name}",
        y_true_encoded=y_test,
        y_pred_encoded=y_pred,
        label_encoder=label_encoder,
    )
    metrics["embedding_model"] = EMBEDDING_MODEL_NAME
    metrics["classifier"] = model_name
    metrics["best_params"] = grid.best_params_
    metrics["cv_macro_f1"] = grid.best_score_

    tuning_results.append(metrics)
    best_estimators[model_name] = best_model

tuning_df = pd.DataFrame(tuning_results).sort_values(
    "macro_f1",
    ascending=False,
)

tuning_df

Tuning LogisticRegression
Fitting 5 folds for each of 10 candidates, totalling 50 fits
IT Role Classifier: BGE + tuned LogisticRegression
{'model_name': 'IT Role Classifier: BGE + tuned LogisticRegression', 'accuracy': 0.8885017421602788, 'macro_f1': 0.8996582226611061, 'weighted_f1': 0.8882683014895919, 'precision_macro': 0.9155063729943748, 'recall_macro': 0.8875139526673479}

                         precision    recall  f1-score   support

  Blockchain Developer       1.00      1.00      1.00        10
   Business Analyst IT       1.00      1.00      1.00        10
         Data Engineer       1.00      0.90      0.95        10
        Data Scientist       1.00      0.70      0.82        10
Database Administrator       0.87      0.88      0.88       186
       DevOps Engineer       1.00      0.90      0.95        10
      DotNet Developer       1.00      1.00      1.00        10
    Frontend Developer       0.74      0.74      0.74       105
        Java Developer       0.98      0

,model_name,accuracy,macro_f1,weighted_f1,precision_macro,recall_macro,embedding_model,classifier,best_params,cv_macro_f1
1,IT Role Classifier: BGE + tuned LinearSVC,0.892567,0.905637,0.891943,0.919082,0.895735,BAAI/bge-small-en-v1.5,LinearSVC,"{'C': 3, 'class_weight': None, 'loss': 'square...",0.882119
0,IT Role Classifier: BGE + tuned LogisticRegres...,0.888502,0.899658,0.888268,0.915506,0.887514,BAAI/bge-small-en-v1.5,LogisticRegression,"{'C': 30, 'class_weight': None}",0.886634


## Выбор лучшего классификатора


In [29]:
best_row = tuning_df.iloc[0]

best_classifier_name = best_row["classifier"]
best_test_metrics = best_row.to_dict()
best_classifier = best_estimators[best_classifier_name]

print("Best classifier:", best_classifier_name)
print("Best params:", best_row["best_params"])
print("Best test metrics:")
print(best_test_metrics)

Best classifier: LinearSVC
Best params: {'C': 3, 'class_weight': None, 'loss': 'squared_hinge'}
Best test metrics:
{'model_name': 'IT Role Classifier: BGE + tuned LinearSVC', 'accuracy': 0.8925667828106852, 'macro_f1': 0.9056374529947013, 'weighted_f1': 0.8919431742243068, 'precision_macro': 0.9190816324419396, 'recall_macro': 0.8957346719103965, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'classifier': 'LinearSVC', 'best_params': {'C': 3, 'class_weight': None, 'loss': 'squared_hinge'}, 'cv_macro_f1': 0.8821189101685695}


## Cross-validation лучшего классификатора


In [15]:
X_all_emb = encode_texts(embedding_model, texts)

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

In [30]:
def cross_validate(
    X_emb: np.ndarray,
    y: np.ndarray,
    classifier,
    model_name: str,
    n_splits: int = 5,
) -> tuple[pd.DataFrame, dict]:
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42,
    )

    rows = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_emb, y), start=1):
        fold_classifier = clone(classifier)

        fold_classifier.fit(X_emb[train_idx], y[train_idx])
        y_pred = fold_classifier.predict(X_emb[test_idx])

        rows.append(
            {
                "model_name": model_name,
                "fold": fold,
                "accuracy": accuracy_score(y[test_idx], y_pred),
                "macro_f1": f1_score(y[test_idx], y_pred, average="macro"),
                "weighted_f1": f1_score(y[test_idx], y_pred, average="weighted"),
                "precision_macro": precision_score(
                    y[test_idx],
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
                "recall_macro": recall_score(
                    y[test_idx],
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
            }
        )

    cv_df = pd.DataFrame(rows)

    summary = {
        "accuracy_mean": cv_df["accuracy"].mean(),
        "accuracy_std": cv_df["accuracy"].std(),
        "macro_f1_mean": cv_df["macro_f1"].mean(),
        "macro_f1_std": cv_df["macro_f1"].std(),
        "weighted_f1_mean": cv_df["weighted_f1"].mean(),
        "weighted_f1_std": cv_df["weighted_f1"].std(),
        "precision_macro_mean": cv_df["precision_macro"].mean(),
        "recall_macro_mean": cv_df["recall_macro"].mean(),
    }

    return cv_df, summary


cv_results_df, cv_metrics = cross_validate(
    X_emb=X_all_emb,
    y=y,
    classifier=best_classifier,
    model_name=f"IT Role Classifier: BGE + {best_classifier_name}",
    n_splits=5,
)

print("CV summary:")
print(cv_metrics)

cv_results_df

CV summary:
{'accuracy_mean': np.float64(0.8765100915722364), 'accuracy_std': 0.008014321490003461, 'macro_f1_mean': np.float64(0.8814890213190294), 'macro_f1_std': 0.017329875874964547, 'weighted_f1_mean': np.float64(0.8753575280318895), 'weighted_f1_std': 0.008346495350372575, 'precision_macro_mean': np.float64(0.8976184190839707), 'recall_macro_mean': np.float64(0.8695509920867334)}


,model_name,fold,accuracy,macro_f1,weighted_f1,precision_macro,recall_macro
0,IT Role Classifier: BGE + LinearSVC,1,0.866434,0.868146,0.865141,0.893783,0.848948
1,IT Role Classifier: BGE + LinearSVC,2,0.886760,0.903101,0.886634,0.912084,0.895153
2,IT Role Classifier: BGE + LinearSVC,3,0.877468,0.896375,0.876008,0.907222,0.887806
3,IT Role Classifier: BGE + LinearSVC,4,0.880883,0.875679,0.879257,0.889023,0.866923
4,IT Role Classifier: BGE + LinearSVC,5,0.871005,0.864144,0.869747,0.885981,0.848926


## Финальное обучение на всех данных

In [31]:
final_classifier = clone(best_classifier)
final_classifier.fit(X_all_emb, y)

print("Final classifier trained on all data")
print("Classifier:", best_classifier_name)


Final classifier trained on all data
Classifier: LinearSVC


## Сохранение модели


In [32]:
final_classifier = best_classifier.__class__(**best_classifier.get_params())
final_classifier.fit(X_all_emb, y)

joblib.dump(final_classifier, MODEL_DIR / "embedding_classifier.pkl")
joblib.dump(label_encoder, MODEL_DIR / "label_encoder.pkl")

metadata = {
    "model_type": "it_role_classifier",
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "classifier_name": best_classifier_name,
    "params": final_classifier.get_params(),
    "classes": label_encoder.classes_.tolist(),
    "test_metrics": best_test_metrics,
    "cv_metrics": cv_metrics,
    "data": {
        "dataset_path": str(DATA_PATH),
        "rows": len(df),
        "sources": df["source"].value_counts().to_dict(),
        "class_distribution": df[TARGET_COLUMN].value_counts().to_dict(),
    },
}

joblib.dump(metadata, MODEL_DIR / "metadata.pkl")

['/content/app/models/it_role_model/metadata.pkl']

## Тестовое предсказание

In [33]:
test_resume = """
Data Engineer with experience in Python, SQL, Apache Spark, PySpark, Airflow, Hadoop, Hive, HDFS, ETL pipelines, data warehouses and data marts.
Built reliable data pipelines, optimized Spark jobs, created data quality checks and automated batch processing workflows.
"""

test_emb = embedding_model.encode(
    [test_resume],
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=device,
)

test_pred = final_classifier.predict(test_emb)
test_role = label_encoder.inverse_transform(test_pred)[0]

print("Predicted role:", test_role)


Predicted role: Data Engineer
